In [15]:
import re

In [16]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total characters:", len(raw_text))
print("\nFirst 100 characters:")
print(raw_text[:100])

Total characters: 20479

First 100 characters:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [17]:
preprocessed = re.split(
    r'([,.?_!"()\']|--|\s)',
    raw_text
)
preprocessed = [
    item.strip()
    for item in preprocessed
    if item.strip()
]


print("Total tokens:", len(preprocessed))
print("\nFirst 30 tokens:")
print(preprocessed[:30])

Total tokens: 4649

First 30 tokens:
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [18]:
all_words = sorted(list(set(preprocessed)))

vocab_size = len(all_words)

print("Vocabulary size:", vocab_size)

Vocabulary size: 1159


In [19]:
vocab = {
    token: integer
    for integer, token in enumerate(all_words)
}

print("First 10 vocabulary entries:")

for i, item in enumerate(vocab.items()):
    print(item)

    if i >= 9:
        break

First 10 vocabulary entries:
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)


In [20]:
class SimpleTokenizerV1:

    def __init__(self, vocab):
        self.str_to_int = vocab

        self.int_to_str = {
            i: s for s, i in vocab.items()
        }

    def encode(self, text):

        preprocessed = re.split(
            r'([,.?_!"()\']|--|\s)',
            text
        )

        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip()
        ]

        ids = [
            self.str_to_int[s]
            for s in preprocessed
        ]

        return ids

    def decode(self, ids):

        text = " ".join(
            [self.int_to_str[i] for i in ids]
        )

        text = re.sub(
            r'\s+([,.?!"()\'])',
            r'\1',
            text
        )

        return text

In [21]:
tokenizer_v1 = SimpleTokenizerV1(vocab)

text = "It's the last he painted, you know."

ids = tokenizer_v1.encode(text)

print("Original text:")
print(text)

print("\nToken IDs:")
print(ids)

decoded_text = tokenizer_v1.decode(ids)

print("\nDecoded text:")
print(decoded_text)

Original text:
It's the last he painted, you know.

Token IDs:
[58, 2, 872, 1013, 615, 541, 763, 5, 1155, 608, 7]

Decoded text:
It' s the last he painted, you know.


In [22]:
new_text = "Hello, do you like tea?"

try:
    print(tokenizer_v1.encode(new_text))

except KeyError as e:
    print("KeyError:", e)

KeyError: 'Hello'


In [23]:
all_tokens = sorted(list(set(preprocessed)))

all_tokens.extend([
    "<|endoftext|>",
    "<|unk|>"
])

vocab = {
    token: integer
    for integer, token in enumerate(all_tokens)
}

print("Updated vocabulary size:", len(vocab))

Updated vocabulary size: 1161


In [24]:
print("Last 5 vocabulary entries:")

for item in list(vocab.items())[-5:]:
    print(item)

Last 5 vocabulary entries:
('younger', 1156)
('your', 1157)
('yourself', 1158)
('<|endoftext|>', 1159)
('<|unk|>', 1160)


In [25]:
class SimpleTokenizerV2:

    def __init__(self, vocab):
        self.str_to_int = vocab

        self.int_to_str = {
            i: s for s, i in vocab.items()
        }

    def encode(self, text):

        preprocessed = re.split(
            r'([,.?_!"()\']|--|\s)',
            text
        )

        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>"
            for item in preprocessed
        ]

        ids = [
            self.str_to_int[s]
            for s in preprocessed
        ]

        return ids

    def decode(self, ids):

        text = " ".join(
            [self.int_to_str[i] for i in ids]
        )

        text = re.sub(
            r'\s+([,.?!"()\'])',
            r'\1',
            text
        )

        return text

In [26]:
tokenizer = SimpleTokenizerV2(vocab)

print("SimpleTokenizerV2 created successfully!")

SimpleTokenizerV2 created successfully!


In [27]:
text1 = "Hello, do you like tea?"

text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [28]:
ids = tokenizer.encode(text)

print("Token IDs:")
print(ids)

Token IDs:
[1160, 5, 362, 1155, 642, 1000, 10, 1159, 57, 1013, 981, 1009, 738, 1013, 1160, 7]


In [29]:
decoded_text = tokenizer.decode(ids)

print("Decoded text:")
print(decoded_text)

Decoded text:
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [30]:
original_text = text

encoded_ids = tokenizer.encode(original_text)

decoded_text = tokenizer.decode(encoded_ids)

print("Original text:")
print(original_text)

print("\nEncoded IDs:")
print(encoded_ids)

print("\nDecoded text:")
print(decoded_text)

print("\nEncoding successful:", len(encoded_ids) > 0)
print("Decoding successful:", len(decoded_text) > 0)

Original text:
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

Encoded IDs:
[1160, 5, 362, 1155, 642, 1000, 10, 1159, 57, 1013, 981, 1009, 738, 1013, 1160, 7]

Decoded text:
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.

Encoding successful: True
Decoding successful: True


In [31]:
print("Total unique tokens:", len(all_words))
print("Updated vocabulary size:", len(vocab))

print("\nSpecial token IDs:")

print("<|endoftext|>:", vocab["<|endoftext|>"])
print("<|unk|>:", vocab["<|unk|>"])

Total unique tokens: 1159
Updated vocabulary size: 1161

Special token IDs:
<|endoftext|>: 1159
<|unk|>: 1160


**BPE**

In [32]:
!pip install tiktoken

In [33]:
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


In [34]:
tokenizer = tiktoken.get_encoding("gpt2")

print("GPT-2 tokenizer loaded!")

GPT-2 tokenizer loaded!


In [35]:
text = (
    "Hello, do you like tea? "
    "<|endoftext|> "
    "In the sunlit terraces of someunknownPlace"
)

integers = tokenizer.encode(
    text,
    allowed_special={"<|endoftext|>"}
)

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271]


In [36]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace


In [37]:
print("Vocabulary size:", tokenizer.n_vocab)

print(
    "End of text ID:",
    tokenizer.encode(
        "<|endoftext|>",
        allowed_special={"<|endoftext|>"}
    )
)

Vocabulary size: 50257
End of text ID: [50256]


In [38]:
text = "Akwirw ier"

ids = tokenizer.encode(text)

print("Token IDs:", ids)

print("\nIndividual tokens:")

for token_id in ids:
    print(token_id, "->", tokenizer.decode([token_id]))

print("\nComplete decoded text:")
print(tokenizer.decode(ids))

Token IDs: [33901, 86, 343, 86, 220, 959]

Individual tokens:
33901 -> Ak
86 -> w
343 -> ir
86 -> w
220 ->  
959 -> ier

Complete decoded text:
Akwirw ier


**Data Sampling with Sliding Window**

In [39]:
import torch
from torch.utils.data import Dataset, DataLoader

In [40]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)

print("Total tokens:", len(enc_text))

Total tokens: 5145


In [41]:
enc_sample = enc_text[50:]

print("Tokens after removing first 50:", len(enc_sample))
print("First 10 tokens:", enc_sample[:10])

Tokens after removing first 50: 5095
First 10 tokens: [290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686]


In [42]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print("Input (x):", x)
print("Target (y):", y)

Input (x): [290, 4920, 2241, 287]
Target (y): [4920, 2241, 287, 257]


In [43]:
for i in range(1, context_size + 1):

    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [44]:
for i in range(1, context_size + 1):

    context = enc_sample[:i]
    desired = enc_sample[i]

    print(
        tokenizer.decode(context),
        "---->",
        tokenizer.decode([desired])
    )

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [45]:
class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):

        self.tokenizer = tokenizer

        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):

            input_chunk = token_ids[i:i + max_length]

            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))

            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):

        return len(self.input_ids)

    def __getitem__(self, idx):

        return self.input_ids[idx], self.target_ids[idx]

In [46]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True
):

    dataset = GPTDatasetV1(
        txt,
        tokenizer,
        max_length,
        stride
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last
    )

    return dataloader

In [47]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)

print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [48]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)

print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [49]:
second_batch = next(data_iter)

print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [50]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=2,
    stride=2,
    shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)

print(first_batch)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]


In [51]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=8,
    stride=2,
    shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)

print(first_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]


In [52]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

data_iter = iter(dataloader)

inputs, targets = next(data_iter)

print("Inputs:\n", inputs)

print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [53]:
print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)

Inputs shape: torch.Size([8, 4])
Targets shape: torch.Size([8, 4])


**Creating Token Embeddings**

In [54]:

# 2.7 Creating Token Embeddings

import torch

# Step 1: Input token IDs
input_ids = torch.tensor([2, 3, 5, 1])
print("Input IDs:", input_ids)

# Step 2: Define vocabulary size and embedding dimensions
vocab_size = 6
output_dim = 3

# Step 3: Initialize embedding layer
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# Step 4: Display embedding weight matrix
print("\nEmbedding Weight Matrix:")
print(embedding_layer.weight)

# Step 5: Get embedding for one token ID
print("\nEmbedding for token ID 3:")
print(embedding_layer(torch.tensor([3])))

# Step 6: Get embeddings for all input IDs
print("\nEmbeddings for all input IDs:")
embedded = embedding_layer(input_ids)
print(embedded)

# Step 7: Check output shape
print("\nOutput Shape:", embedded.shape)


Input IDs: tensor([2, 3, 5, 1])

Embedding Weight Matrix:
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

Embedding for token ID 3:
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

Embeddings for all input IDs:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

Output Shape: torch.Size([4, 3])


*2.8 Encoding Word Positions***

In [55]:

# 2.8 Encoding Word Positions

# Step 1: Define vocabulary and embedding dimensions
vocab_size = 50257
output_dim = 256

# Step 2: Create token embedding layer
token_embedding_layer = torch.nn.Embedding(
    vocab_size,
    output_dim
)

# Step 3: Create DataLoader
max_length = 4

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length,
    shuffle=False
)

# Step 4: Get one batch
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs:\n", inputs)
print("Inputs shape:", inputs.shape)

# Step 5: Convert token IDs into token embeddings
token_embeddings = token_embedding_layer(inputs)

print("\nToken embeddings shape:")
print(token_embeddings.shape)

# Step 6: Create positional embedding layer
context_length = max_length

pos_embedding_layer = torch.nn.Embedding(
    context_length,
    output_dim
)

# Step 7: Generate position IDs and their embeddings
position_ids = torch.arange(context_length)

pos_embeddings = pos_embedding_layer(position_ids)

print("\nPosition IDs:", position_ids)
print("Positional embeddings shape:", pos_embeddings.shape)

# Step 8: Add token and positional embeddings
input_embeddings = token_embeddings + pos_embeddings

print("\nFinal input embeddings shape:")
print(input_embeddings.shape)


Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Inputs shape: torch.Size([8, 4])

Token embeddings shape:
torch.Size([8, 4, 256])

Position IDs: tensor([0, 1, 2, 3])
Positional embeddings shape: torch.Size([4, 256])

Final input embeddings shape:
torch.Size([8, 4, 256])
